# What is in front of the camera?

The camera shows what it sees. Tap **Take a shot**: the objects in the photo are found in your browser by an object-detection model (`detr-resnet-50`, about 43 MB), and the photo never leaves this page. Tap **Retry** to go back to the camera. The first visit downloads the model, so give it a moment.

The model names the 80 COCO object classes (people, cats, cups, laptops, chairs and so on) and it is not always right. This is an app built with [lmLite](https://github.com/decitre/lmlite): [the notebook behind it](https://decitre.github.io/lmlite/xeus/lab/index.html?path=ch15_camera.ipynb).

In [ ]:
import asyncio
import io

import ipywidgets as widgets
from IPython.display import display
from PIL import Image as PilImage
from PIL import ImageDraw, ImageFont

from lmlite import pipeline
from lmlite.camera import CameraCapture

FONT = ImageFont.load_default_imagefont()
COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#008080", "#9a6324", "#800000"]


def badge(text, color, scale):
    left, top, right, bottom = FONT.getbbox(text)
    mask = PilImage.new("L", (right, bottom), 0)
    ImageDraw.Draw(mask).text((0, 0), text, font=FONT, fill=255)
    mask = mask.resize((mask.width * scale, mask.height * scale), PilImage.NEAREST)
    strip = PilImage.new("RGB", (mask.width + 8, mask.height + 8), color)
    strip.paste("white", (4, 4), mask)
    return strip


def annotate(image, detections):
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    colors = {}
    for item in detections:
        box = item["box"]
        color = colors.setdefault(item["label"], COLORS[len(colors) % len(COLORS)])
        draw.rectangle((box["xmin"], box["ymin"], box["xmax"], box["ymax"]), outline=color, width=4)
        strip = badge(f"{item['label']} {item['score']:.2f}", color, scale=2)
        annotated.paste(strip, (round(box["xmin"]), max(0, round(box["ymin"]) - strip.height)))
    return annotated


# The camera shows its picture and nothing else: the button below takes the shot.
camera = CameraCapture(controls=False, autostart=True, show_photo=False)
button = widgets.Button(description="Loading the model...", disabled=True, button_style="primary")
status = widgets.Label()
picture = widgets.Image(format="png", layout=widgets.Layout(max_width="100%", display="none"))
detector = None
state = "loading"  # loading, live, detecting or result


def show(new_state):
    global state
    state = new_state
    camera.layout.display = "" if new_state in ("live", "detecting") else "none"
    picture.layout.display = "" if new_state == "result" else "none"
    button.description = {
        "loading": "Loading the model...",
        "live": "Take a shot",
        "detecting": "Looking for objects...",
        "result": "Retry",
    }[new_state]
    button.disabled = new_state in ("loading", "detecting") or (new_state == "live" and not camera.running)


def on_click(_):
    if state == "live":
        camera.take()
    elif state == "result":
        camera.clear()
        status.value = ""
        show("live")


async def detect(jpeg):
    try:
        detections = await detector(jpeg, threshold=0.6)
        photo = PilImage.open(io.BytesIO(jpeg)).convert("RGB")
        png = io.BytesIO()
        annotate(photo, detections).save(png, format="PNG")
        picture.value = png.getvalue()
        found = ", ".join(f"{item['label']} {item['score']:.2f}" for item in detections)
        status.value = f"Found: {found}" if found else "No object found."
        show("result")
    except Exception as error:
        status.value = f"Failed: {error}"
        show("live")


def on_photo(change):
    if change["new"] and state == "live":
        show("detecting")
        asyncio.create_task(detect(camera.image))


button.on_click(on_click)
camera.observe(on_photo, names="photo")
camera.observe(lambda change: show(state), names="running")  # the button follows the camera
display(widgets.VBox([widgets.HBox([button, status]), camera, picture]))
detector = await pipeline("object-detection", model="detr-resnet-50", backend="webgpu", quiet=True)
show("live")